In [6]:
# Задание 2: Квантовая схема булевой функции — Вариант 12
# F(A, B, C, D, E) = MAJ(A, B, C) ⊕ MAJ(C, D, E)

from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_aer import Aer
from qiskit.quantum_info import Statevector
import numpy as np
from itertools import product

print("ЗАДАНИЕ 2: КВАНТОВАЯ СХЕМА БУЛЕВОЙ ФУНКЦИИ — ВАРИАНТ 12")
print("F(A, B, C, D, E) = MAJ(A, B, C) ⊕ MAJ(C, D, E)")

# 2.1 Классическая реализация F на чистом Python

print("\n--- 2.1 Классическая реализация F ---")


def maj(a, b, c):
    """Мажоритарная функция: возвращает 1, если большинство входов = 1"""
    return (a & b) | (b & c) | (a & c)


def F_classical(A, B, C, D, E):
    """F(A, B, C, D, E) = MAJ(A, B, C) ⊕ MAJ(C, D, E)"""
    return maj(A, B, C) ^ maj(C, D, E)


# Таблица истинности
print("\nТаблица истинности F(A,B,C,D,E) = MAJ(A,B,C) ⊕ MAJ(C,D,E):")
print(f"{'A':>2} {'B':>2} {'C':>2} {'D':>2} {'E':>2} | {'MAJ1':>4} {'MAJ2':>4} | {'F':>2}")
print("-" * 40)
for A, B, C, D, E in product([0, 1], repeat=5):
    m1 = maj(A, B, C)
    m2 = maj(C, D, E)
    f = m1 ^ m2
    print(f"{A:>2} {B:>2} {C:>2} {D:>2} {E:>2} | {m1:>4} {m2:>4} | {f:>2}")



# 2.2 Базовые элементы как подсхемы

print("\n--- 2.2 Базовые элементы как подсхемы ---")


def apply_and(qc, control1, control2, target):
    """AND: target ^= (control1 & control2) через CCX"""
    qc.ccx(control1, control2, target)


def apply_xor_to_target(qc, control, target):
    """XOR: target ^= control через CX"""
    qc.cx(control, target)


def apply_maj(qc, a, b, c, target, anc1, anc2, anc3):
    """
    MAJ(a,b,c) = (a∧b) ⊕ (b∧c) ⊕ (a∧c)
    Вычисляется в target кубит через ancilla anc1, anc2, anc3
    anc1 = a∧b, anc2 = b∧c, anc3 = a∧c
    target ^= anc1 ⊕ anc2 ⊕ anc3
    """
    # Вычисляем попарные AND в ancilla
    qc.ccx(a, b, anc1)    # anc1 = A∧B
    qc.ccx(b, c, anc2)    # anc2 = B∧C
    qc.ccx(a, c, anc3)    # anc3 = A∧C

    # XOR всех трёх в target
    qc.cx(anc1, target)
    qc.cx(anc2, target)
    qc.cx(anc3, target)


def uncompute_maj(qc, a, b, c, target, anc1, anc2, anc3):
    """
    Обратное вычисление MAJ: очищает ancilla кубиты
    """
    # Обратный XOR
    qc.cx(anc3, target)
    qc.cx(anc2, target)
    qc.cx(anc1, target)

    # Обратные CCX (они сами обратны)
    qc.ccx(a, c, anc3)
    qc.ccx(b, c, anc2)
    qc.ccx(a, b, anc1)



# 2.3 Базовая версия схемы (каждый промежуточный результат в отдельный ancilla)

print("\n--- 2.3 Базовая версия схемы ---")


def build_F_basic():
    """
    Базовая версия: каждый промежуточный результат в отдельный ancilla кубит.
    Кубиты: 0=A, 1=B, 2=C, 3=D, 4=E, 5=F(выход), 6-11=ancilla
    anc6 = A∧B, anc7 = B∧C, anc8 = A∧C (для MAJ1)
    anc9 = C∧D, anc10 = D∧E, anc11 = C∧E (для MAJ2)
    """
    qc = QuantumCircuit(12, 1)

    A, B, C, D, E = 0, 1, 2, 3, 4
    F_out = 5
    anc1, anc2, anc3 = 6, 7, 8       # для MAJ(A,B,C)
    anc4, anc5, anc6 = 9, 10, 11     # для MAJ(C,D,E)

    # Вычисляем MAJ(A,B,C) в F_out
    apply_maj(qc, A, B, C, F_out, anc1, anc2, anc3)

    # Вычисляем MAJ(C,D,E) и XOR в F_out
    apply_maj(qc, C, D, E, F_out, anc4, anc5, anc6)

    # Измеряем выход
    qc.measure(F_out, 0)

    # Uncompute MAJ(C,D,E)
    uncompute_maj(qc, C, D, E, F_out, anc4, anc5, anc6)

    # Uncompute MAJ(A,B,C)
    uncompute_maj(qc, A, B, C, F_out, anc1, anc2, anc3)

    return qc


qc_basic = build_F_basic()
print(f"Базовая схема:")
print(f"  Количество кубитов: {qc_basic.num_qubits}")
print(f"  Глубина: {qc_basic.depth()}")
print(f"  Количество CCX: {sum(1 for op in qc_basic.data if op[0].name == 'ccx')}")
print(f"  Количество CX: {sum(1 for op in qc_basic.data if op[0].name == 'cx')}")



# 2.4 Оптимизированная версия схемы

print("\n--- 2.4 Оптимизированная версия схемы ---")


def build_F_optimized():
    """
    Оптимизированная версия: переиспользуем ancilla кубиты.
    Кубиты: 0=A, 1=B, 2=C, 3=D, 4=E, 5=F(выход), 6-8=ancilla (переиспользуемые)
    """
    qc = QuantumCircuit(9, 1)

    A, B, C, D, E = 0, 1, 2, 3, 4
    F_out = 5
    anc1, anc2, anc3 = 6, 7, 8

    # Вычисляем MAJ(A,B,C) в F_out
    apply_maj(qc, A, B, C, F_out, anc1, anc2, anc3)

    # Uncompute MAJ(A,B,C) — очищаем ancilla для переиспользования
    # Но сначала сохраним результат в F_out, поэтому uncompute только ancilla
    # Обратные CCX очищают ancilla, но target (F_out) уже содержит результат
    qc.ccx(A, C, anc3)
    qc.ccx(B, C, anc2)
    qc.ccx(A, B, anc1)

    # Теперь ancilla чистые, вычисляем MAJ(C,D,E)
    apply_maj(qc, C, D, E, F_out, anc1, anc2, anc3)

    # Uncompute MAJ(C,D,E) — очищаем ancilla
    qc.ccx(C, E, anc3)
    qc.ccx(D, E, anc2)
    qc.ccx(C, D, anc1)

    # Измеряем выход
    qc.measure(F_out, 0)

    return qc


qc_opt = build_F_optimized()
print(f"Оптимизированная схема:")
print(f"  Количество кубитов: {qc_opt.num_qubits}")
print(f"  Глубина: {qc_opt.depth()}")
print(f"  Количество CCX: {sum(1 for op in qc_opt.data if op[0].name == 'ccx')}")
print(f"  Количество CX: {sum(1 for op in qc_opt.data if op[0].name == 'cx')}")



# 2.5 Сравнение базовой и оптимизированной схем

print("\n--- 2.5 Сравнение схем ---")
print(f"{'Параметр':<20} {'Базовая':<10} {'Оптимиз.':<10}")
print("-" * 40)
print(f"{'Кубиты':<20} {qc_basic.num_qubits:<10} {qc_opt.num_qubits:<10}")
print(f"{'CCX вентили':<20} {sum(1 for op in qc_basic.data if op[0].name=='ccx'):<10} {sum(1 for op in qc_opt.data if op[0].name=='ccx'):<10}")
print(f"{'CX вентили':<20} {sum(1 for op in qc_basic.data if op[0].name=='cx'):<10} {sum(1 for op in qc_opt.data if op[0].name=='cx'):<10}")
print(f"{'Глубина':<20} {qc_basic.depth():<10} {qc_opt.depth():<10}")

print("""
Объяснение: Оптимизированная схема использует меньше кубитов (9 вместо 12),
переиспользуя ancilla кубиты. Однако это может увеличить глубину схемы,
так как операции выполняются последовательно, а не параллельно.
Минимум по числу кубитов и минимум по глубине могут не совпадать,
потому что увеличение числа кубитов позволяет параллелизовать вычисления.
""")



# 2.6 Проверка на всех 2^5 = 32 входах

print("\n--- 2.6 Проверка квантовой схемы на всех 32 входах ---")

backend = Aer.get_backend('statevector_simulator')

results_table = []
all_match = True
all_ancilla_zero = True

for idx in range(32):
    A = (idx >> 4) & 1
    B = (idx >> 3) & 1
    C = (idx >> 2) & 1
    D = (idx >> 1) & 1
    E = idx & 1

    # Классическое значение
    f_classical = F_classical(A, B, C, D, E)

    # Квантовая схема (оптимизированная)
    qc_test = QuantumCircuit(9, 1)

    # Устанавливаем входы
    if A: qc_test.x(0)
    if B: qc_test.x(1)
    if C: qc_test.x(2)
    if D: qc_test.x(3)
    if E: qc_test.x(4)

    # Вычисляем MAJ(A,B,C) в F_out
    apply_maj(qc_test, 0, 1, 2, 5, 6, 7, 8)

    # Uncompute ancilla
    qc_test.ccx(0, 2, 8)
    qc_test.ccx(1, 2, 7)
    qc_test.ccx(0, 1, 6)

    # Вычисляем MAJ(C,D,E) и XOR в F_out
    apply_maj(qc_test, 2, 3, 4, 5, 6, 7, 8)

    # Uncompute ancilla
    qc_test.ccx(2, 4, 8)
    qc_test.ccx(3, 4, 7)
    qc_test.ccx(2, 3, 6)

    # Получаем statevector ДО измерения
    sv = Statevector(qc_test)
    probs = sv.probabilities_dict()

    # Проверяем ancilla кубиты (должны быть |0>)
    # ancilla кубиты: 6, 7, 8; входные: 0-4; выход: 5
    # Проверяем, что ancilla в состоянии |0>
    for state_key, prob in probs.items():
        if prob > 0.01:  # Значимая вероятность
            # state_key в формате 'q8 q7 q6 q5 q4 q3 q2 q1 q0'
            bits = state_key.split()
            # ancilla: bits[0]=q8, bits[1]=q7, bits[2]=q6
            if bits[0] != '0' or bits[1] != '0' or bits[2] != '0':
                all_ancilla_zero = False

    # Измеряем
    qc_test.measure(5, 0)
    job = backend.run(qc_test, shots=100)
    counts = job.result().get_counts()

    # Определяем квантовый результат
    f_quantum = int(list(counts.keys())[0][0]) if counts else 0

    match = "✓" if f_quantum == f_classical else "✗"
    if f_quantum != f_classical:
        all_match = False

    results_table.append((A, B, C, D, E, f_classical, f_quantum, match))

print(f"\n{'A':>2} {'B':>2} {'C':>2} {'D':>2} {'E':>2} | {'F_класс':>7} {'F_квант':>7} | {'Статус':>6}")
print("-" * 45)
for row in results_table:
    print(f"{row[0]:>2} {row[1]:>2} {row[2]:>2} {row[3]:>2} {row[4]:>2} | {row[5]:>7} {row[6]:>7} | {row[7]:>6}")

print(f"\nВсе совпадения: {all_match}")
print(f"Ancilla кубиты вернулись в |0⟩: {all_ancilla_zero}")

# 2.7 Проверка через shots=1000 для надёжности

print("\n--- 2.7 Проверка с shots=1000 ---")

backend_qasm = Aer.get_backend('qasm_simulator')
correct_count = 0
total_tests = 32

for idx in range(32):
    A = (idx >> 4) & 1
    B = (idx >> 3) & 1
    C = (idx >> 2) & 1
    D = (idx >> 1) & 1
    E = idx & 1

    f_classical = F_classical(A, B, C, D, E)

    qc_test = QuantumCircuit(9, 1)
    if A: qc_test.x(0)
    if B: qc_test.x(1)
    if C: qc_test.x(2)
    if D: qc_test.x(3)
    if E: qc_test.x(4)

    apply_maj(qc_test, 0, 1, 2, 5, 6, 7, 8)
    qc_test.ccx(0, 2, 8)
    qc_test.ccx(1, 2, 7)
    qc_test.ccx(0, 1, 6)
    apply_maj(qc_test, 2, 3, 4, 5, 6, 7, 8)
    qc_test.ccx(2, 4, 8)
    qc_test.ccx(3, 4, 7)
    qc_test.ccx(2, 3, 6)

    qc_test.measure(5, 0)
    job = backend_qasm.run(qc_test, shots=1000)
    counts = job.result().get_counts()

    # Majority vote
    f_quantum = max(counts, key=counts.get)
    f_quantum = int(f_quantum[0])

    if f_quantum == f_classical:
        correct_count += 1

accuracy = correct_count / total_tests
print(f"Точность: {correct_count}/{total_tests} = {accuracy:.1%}")
print(f"Результат: {'✓ ВСЕ ВЕРНО' if accuracy == 1.0 else '✗ ЕСТЬ ОШИБКИ'}")

ЗАДАНИЕ 2: КВАНТОВАЯ СХЕМА БУЛЕВОЙ ФУНКЦИИ — ВАРИАНТ 12
F(A, B, C, D, E) = MAJ(A, B, C) ⊕ MAJ(C, D, E)

--- 2.1 Классическая реализация F ---

Таблица истинности F(A,B,C,D,E) = MAJ(A,B,C) ⊕ MAJ(C,D,E):
 A  B  C  D  E | MAJ1 MAJ2 |  F
----------------------------------------
 0  0  0  0  0 |    0    0 |  0
 0  0  0  0  1 |    0    0 |  0
 0  0  0  1  0 |    0    0 |  0
 0  0  0  1  1 |    0    1 |  1
 0  0  1  0  0 |    0    0 |  0
 0  0  1  0  1 |    0    1 |  1
 0  0  1  1  0 |    0    1 |  1
 0  0  1  1  1 |    0    1 |  1
 0  1  0  0  0 |    0    0 |  0
 0  1  0  0  1 |    0    0 |  0
 0  1  0  1  0 |    0    0 |  0
 0  1  0  1  1 |    0    1 |  1
 0  1  1  0  0 |    1    0 |  1
 0  1  1  0  1 |    1    1 |  0
 0  1  1  1  0 |    1    1 |  0
 0  1  1  1  1 |    1    1 |  0
 1  0  0  0  0 |    0    0 |  0
 1  0  0  0  1 |    0    0 |  0
 1  0  0  1  0 |    0    0 |  0
 1  0  0  1  1 |    0    1 |  1
 1  0  1  0  0 |    1    0 |  1
 1  0  1  0  1 |    1    1 |  0
 1  0  1  1  0 |    1

/tmp/ipykernel_2936/807424791.py:130: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  print(f"  Количество CCX: {sum(1 for op in qc_basic.data if op[0].name == 'ccx')}")
/tmp/ipykernel_2936/807424791.py:131: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  print(f"  Количество CX: {sum(1 for op in qc_basic.data if op[0].name == 'cx')}")
/tmp/ipykernel_2936/807424791.py:179: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  print(f"  Количество CCX: {sum(1 for op in qc_opt.data if op[0].na


 A  B  C  D  E | F_класс F_квант | Статус
---------------------------------------------
 0  0  0  0  0 |       0       0 |      ✓
 0  0  0  0  1 |       0       0 |      ✓
 0  0  0  1  0 |       0       0 |      ✓
 0  0  0  1  1 |       1       1 |      ✓
 0  0  1  0  0 |       0       0 |      ✓
 0  0  1  0  1 |       1       1 |      ✓
 0  0  1  1  0 |       1       1 |      ✓
 0  0  1  1  1 |       1       1 |      ✓
 0  1  0  0  0 |       0       0 |      ✓
 0  1  0  0  1 |       0       0 |      ✓
 0  1  0  1  0 |       0       0 |      ✓
 0  1  0  1  1 |       1       1 |      ✓
 0  1  1  0  0 |       1       1 |      ✓
 0  1  1  0  1 |       0       0 |      ✓
 0  1  1  1  0 |       0       0 |      ✓
 0  1  1  1  1 |       0       0 |      ✓
 1  0  0  0  0 |       0       0 |      ✓
 1  0  0  0  1 |       0       0 |      ✓
 1  0  0  1  0 |       0       0 |      ✓
 1  0  0  1  1 |       1       1 |      ✓
 1  0  1  0  0 |       1       1 |      ✓
 1  0  1  0  1 |       0     